# Módulo 2 — Fase de Descenso (Descent)
**Programa:** Inspira STEM 2026  
**Instructores:** Oscar Tejada y Patricia Ortíz

---
## Dinámica de Paracaídas Supersónicos
Aproximadamente a 10 km de altitud, el vehículo debe transicionar del régimen hipersónico al subsónico. En la misión real, el sistema de navegación del Curiosity dictó el despliegue de un paracaídas de 16 metros de diámetro mientras la cápsula aún viajaba a cerca de Mach 1.7 (más de 1,500 km/h).

El diseño de este subsistema implica un compromiso (*trade-off*) aerodinámico crítico: un área de arrastre mayor genera una desaceleración cinemática más eficiente, pero induce un choque térmico y mecánico instantáneo al momento de la apertura que puede comprometer la integridad estructural de las líneas de Kevlar.

---

### Paso 1: Configuración del Entorno de Simulación
Ejecuta la celda para compilar las ecuaciones de mecánica de fluidos y asegura el ingreso correcto de las variables del entorno atmosférico asignado a tu equipo.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact

plt.rcParams.update({'figure.dpi': 100, 'font.size': 10, 'axes.grid': True, 'grid.alpha': 0.3, 'lines.linewidth': 2})

# ==========================================
# ⚙️ DATOS DEL PLANETA ASIGNADO
# ==========================================
g_planeta = 3.711       # Gravedad [m/s^2]
rho_planeta = 0.020     # Densidad superficial [kg/m^3]
h_scale_planeta = 11100 # Escala [m]
a_s_planeta = 240.0     # Velocidad del sonido local [m/s]

def simular_descent(m_ini, cd_p, a_p, m_drop):
    dt, h, v, t, m_act = 0.1, 10000.0, -500.0, 0.0, m_ini
    tel = {"t": [], "h": [], "v": [], "mach": [], "shock": [], "v_term": [], "acel": []}
    while h >= 1800.0 and t < 200:
        if t > 20.0: m_act = m_ini - m_drop 
        rho = rho_planeta * np.exp(-max(h, 0.0) / h_scale_planeta) if rho_planeta > 0 else 0.0
        drag = 0.5 * rho * v**2 * (cd_p * a_p)
        for k, val in zip(["t","h","v","mach","shock","v_term","acel"], 
                          [t, h, abs(v), abs(v)/a_s_planeta, drag/1000, np.sqrt((2*m_act*g_planeta)/max(rho*cd_p*a_p,1e-6)), (drag/m_act)-g_planeta]):
            tel[k].append(val)
        v += ((drag / m_act) - g_planeta) * dt
        h += v * dt
        t += dt
    return {k: np.array(v) for k, v in tel.items()}
print("Túnel de viento computacional inicializado.")

---
### Análisis 1: Tensión Estructural vs. Área del Dosel
Al momento del despliegue supersónico, el paracaídas experimenta un frente de onda de choque masivo. La magnitud de este impacto aerodinámico (Fuerza de Despliegue) está matemáticamente dominada por el **Área ($\mathbf{A_p}$)** del dosel expuesta al flujo dinámico:
$$D_p = \frac{1}{2} \rho(h) v^2 C_{d,p} \mathbf{A_p}$$

La exploración de este parámetro, modificando el Área ($\mathbf{A_p}$), demuestra cómo una superficie geométrica extensa induce una carga mecánica transitoria, la cual debe mantenerse rigurosamente bajo los límites de resistencia a la tracción del ensamble.

In [ ]:
def interact_area_p(area_m2):
    d = simular_descent(m_ini=2400.0, cd_p=0.62, a_p=area_m2, m_drop=0.0)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    axs[0].plot(d["mach"], d["shock"], color="#27ae60")
    axs[0].set(title="Tensión vs Régimen de Mach", xlabel="Número de Mach [M]", ylabel="Fuerza de Impacto [kN]")
    axs[0].invert_xaxis()
    
    axs[1].plot(d["t"], d["shock"], color="#27ae60")
    axs[1].axhline(40.0, color="k", linestyle="--", label="Límite de Suspensión (40 kN)")
    axs[1].set(title="Pico Transitorio de Despliegue", xlabel="Tiempo [s]")
    axs[1].legend()
    plt.show()

interact(interact_area_p, area_m2=widgets.FloatSlider(value=360.0, min=100.0, max=800.0, step=10.0, description='Área [m²]:'));

---
### Análisis 2: Eficiencia Aerodinámica vs. Coeficiente de Forma
Un paracaídas plano convencional colapsa bajo regímenes supersónicos. Curiosity requirió una geometría especializada conocida como "Banda-Disco-Brecha" (Disk-Gap-Band), diseñada para "respirar" el exceso de presión manteniendo la estabilidad.

Esta forma constructiva altera directamente el **Coeficiente de Arrastre ($\mathbf{C_{d,p}}$)** del diseño:
$$D_p = \frac{1}{2} \rho(h) v^2 \mathbf{C_{d,p}} A_p$$

La iteración del Coeficiente de Forma permite modelar cómo la eficiencia geométrica de la tela influye en la tasa de desaceleración general, garantizando que el vehículo alcance el corredor de velocidad necesario para iniciar los propulsores.

In [ ]:
def interact_cd(cd_valor):
    d = simular_descent(m_ini=2400.0, cd_p=cd_valor, a_p=360.0, m_drop=0.0)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    axs[0].plot(d["h"]/1000, d["acel"], color="#2980b9")
    axs[0].set(title="Tasa de Desaceleración por Altitud", xlabel="Altitud [km]", ylabel="Desaceleración [m/s²]")
    axs[0].invert_xaxis()
    
    axs[1].plot(d["t"], d["v"], color="#2980b9")
    axs[1].axhspan(75, 125, color="#f39c12", alpha=0.15, label="Ventana de Separación Nominal")
    axs[1].set(title="Evolución de Velocidad Resultante", xlabel="Tiempo [s]", ylabel="Velocidad [m/s]")
    axs[1].legend()
    plt.show()

interact(interact_cd, cd_valor=widgets.FloatSlider(value=0.62, min=0.2, max=1.2, step=0.05, description='Forma (Cd):'));

---
### Análisis 3: Velocidad Terminal vs. Gestión de Masa
En una atmósfera densa, todo objeto que cae alcanza una Velocidad Terminal ($V_t$), un estado de equilibrio dinámico donde el arrastre aerodinámico anula por completo la aceleración gravitatoria local.

Para desplazar artificialmente este límite asintótico, la misión Curiosity liberó su masivo escudo inferior (aproximadamente 300 kg) en plena caída, reduciendo la **Masa Total ($\mathbf{m_{lastre}}$)** que el paracaídas debía sostener:
$$V_t = \sqrt{\frac{2(m_{inicial} - \mathbf{m_{lastre}})g}{\rho(h) C_{d,p} A_p}}$$

La simulación del evento de separación pirotécnica (modificando la variable de lastre) evidencia matemáticamente el decaimiento inmediato de la asíntota de velocidad, facilitando las condiciones de la fase final.

In [ ]:
def interact_lastre(peso_soltado):
    d = simular_descent(m_ini=2400.0, cd_p=0.62, a_p=360.0, m_drop=peso_soltado)
    fig, axs = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
    
    axs[0].plot(d["h"]/1000, d["v_term"], color="#8e44ad")
    axs[0].set(title="Velocidad Terminal Teórica", xlabel="Altitud [km]", ylabel="V. Terminal [m/s]")
    axs[0].invert_xaxis()
    
    axs[1].plot(d["v"], d["h"]/1000, color="#8e44ad")
    axs[1].scatter(100.0, 1.8, color="gold", marker="D", s=100, edgecolor="k", zorder=5, label="Punto de Separación Radar")
    axs[1].set(title="Perfil de Vuelo y Preparación de Aterrizaje", xlabel="Velocidad [m/s]", ylabel="Altitud [km]")
    axs[1].legend()
    plt.show()

interact(interact_lastre, peso_soltado=widgets.FloatSlider(value=0.0, min=0.0, max=1000.0, step=50.0, description='Lastre [kg]:'));

### Análisis de Grupo
Reúnanse y debatan los siguientes escenarios de ingeniería, utilizando las herramientas interactivas para sustentar sus respuestas:

1. **El Latigazo Supersónico (Misión Real):** En su descenso hacia el cráter Gale, el paracaídas de Curiosity experimentó una carga de apertura repentina equivalente a casi 30 toneladas de fuerza (65,000 libras). Basados en el Análisis 1, detallen el fundamento físico por el cual los ingenieros de JPL no optaron simplemente por diseñar un paracaídas de 50 metros de diámetro para frenar mucho más rápido y evitar riesgos.
2. **El Diseño Geométrico Específico:** Un paracaídas de forma hemisférica estándar, excelente para la Tierra, sufre colapsos dinámicos violentos en el régimen supersónico marciano. Tomando en cuenta la variable del Análisis 2 ($C_d$), ¿de qué manera la eficiencia del diseño del paracaídas dicta la altitud y la velocidad exacta a la que el radar altímetro puede comenzar a rastrear el terreno para cederle el control a los motores?
3. **Soltar el Peso Muerto:** A los 8 kilómetros de altura, Curiosity soltó y dejó caer su pesado escudo térmico inferior. Usando la derivación matemática del límite asintótico en el Análisis 3, expliquen por qué deshacerse de este lastre no tuvo como único objetivo despejar el campo de visión de las cámaras y el radar, sino que alteró permanentemente la física de retención de velocidad de la cápsula.